# 第6章

In [ ]:
!pip install pyro-ppl==1.9.0
!pip install pgmpy==0.1.25
!pip install matplotlib==3.7.1

## リスト6.1

In [ ]:
from pyro.distributions import Normal
from pyro import sample

def cgm_model():
    x = sample("x", Normal(47., 2.3))
    y = sample("y", Normal(25. + 3*x, 3.3))
    return x, y


cgm_model()

## リスト6.2

In [ ]:
from pyro.distributions import Normal
from pyro import sample

def scm_model():
    n_x = sample("n_x", Normal(0., 2.3))
    n_y = sample("n_y", Normal(0., 3.3))
    x = 47. + n_x
    y = 25. + 3.*x + n_y
    return x, y


scm_model()

## リスト6.3

In [ ]:
import pandas as pd
import random

def true_dgp(
    jenny_inclination,
    brian_inclination,
    window_strength):
    jenny_throws_rock = jenny_inclination > 0.5
    brian_throws_rock = brian_inclination > 0.5
    if jenny_throws_rock and brian_throws_rock:
        strength_of_impact = 0.8
    elif jenny_throws_rock or brian_throws_rock:
        strength_of_impact = 0.6
    else:
        strength_of_impact = 0.0
    window_breaks = window_strength < strength_of_impact
    return jenny_throws_rock, brian_throws_rock, window_breaks

generated_outcome = true_dgp(
    jenny_inclination=random.uniform(0, 1),
    brian_inclination=random.uniform(0, 1),
    window_strength=random.uniform(0, 1)
)

generated_outcome

## リスト6.4

In [ ]:
from pgmpy.factors.discrete.CPD import TabularCPD
f_host_door_selection = TabularCPD(
    variable='Host Door Selection',
    variable_card=3,
    values=[
        [0,0,0,0,1,1,0,1,1,0,0,0,0,0,1,0,1,0],
        [1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1],
        [0,1,0,1,0,0,0,0,0,1,1,0,1,1,0,0,0,0]
    ],
    evidence=[
        'Host Inclination',
        'Door with Car',
        'Player First Choice'
    ],
    evidence_card=[2, 3, 3],
    state_names={
        'Host Door Selection':['1st', '2nd', '3rd'],
        'Host Inclination': ['left', 'right'],
        'Door with Car': ['1st', '2nd', '3rd'],
        'Player First Choice': ['1st', '2nd', '3rd']
    }
)

## リスト6.5

In [ ]:
from pgmpy.factors.discrete.CPD import TabularCPD
f_second_choice = TabularCPD(
    variable='Player Second Choice',
    variable_card=3,
    values=[
        [1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,1,0],
        [0,1,0,0,1,0,0,1,0,1,0,1,0,1,0,1,0,1],
        [0,0,1,0,0,1,0,0,1,0,1,0,1,0,0,0,0,0]
    ],
    evidence=[
        'Strategy',
        'Host Door Selection',
        'Player First Choice'
    ],
    evidence_card=[2, 3, 3],
    state_names={
        'Player Second Choice': ['1st', '2nd', '3rd'],
        'Strategy': ['stay', 'switch'],
        'Host Door Selection': ['1st', '2nd', '3rd'],
        'Player First Choice': ['1st', '2nd', '3rd']
    }
)

## リスト6.6

In [ ]:
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete.CPD import TabularCPD

monty_hall_model = BayesianNetwork([
    ('Host Inclination', 'Host Door Selection'),
    ('Door with Car', 'Host Door Selection'),
    ('Player First Choice', 'Host Door Selection'),
    ('Player First Choice', 'Player Second Choice'),
    ('Host Door Selection', 'Player Second Choice'),
    ('Strategy', 'Player Second Choice'),
    ('Player Second Choice', 'Win or Lose'),
    ('Door with Car', 'Win or Lose')
])

## リスト6.7

In [ ]:
p_host_inclination = TabularCPD(
    variable='Host Inclination',
    variable_card=2,
    values=[[.5], [.5]],
    state_names={'Host Inclination': ['left', 'right']}
)

p_door_with_car = TabularCPD(
    variable='Door with Car',
    variable_card=3,
    values=[[1/3], [1/3], [1/3]],
    state_names={'Door with Car': ['1st', '2nd', '3rd']}
)

p_player_first_choice = TabularCPD(
    variable='Player First Choice',
    variable_card=3,
    values=[[1/3], [1/3], [1/3]],
    state_names={'Player First Choice': ['1st', '2nd', '3rd']}
)

p_host_strategy = TabularCPD(
    variable='Strategy',
    variable_card=2,
    values=[[.5], [.5]],
    state_names={'Strategy': ['stay', 'switch']}
)

## リスト6.8

In [ ]:
f_win_or_lose = TabularCPD(
    variable='Win or Lose',
    variable_card=2,
    values=[
        [1,0,0,0,1,0,0,0,1],
        [0,1,1,1,0,1,1,1,0],
    ],
    evidence=['Player Second Choice', 'Door with Car'],
    evidence_card=[3, 3],
    state_names={
        'Win or Lose': ['win', 'lose'],
        'Player Second Choice': ['1st', '2nd', '3rd'],
        'Door with Car': ['1st', '2nd', '3rd']
    }
)

## リスト6.9

In [ ]:
monty_hall_model.add_cpds(
    p_host_inclination,
    p_door_with_car,
    p_player_first_choice,
    p_host_strategy,
    f_host_door_selection,
    f_second_choice,
    f_win_or_lose
)

## リスト6.10

In [ ]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(monty_hall_model)
q1 = infer.query(['Win or Lose'], evidence={'Strategy': 'stay'})
print(q1)
q2 = infer.query(['Win or Lose'], evidence={'Strategy': 'switch'})
print(q2)
q3 = infer.query(['Strategy'], evidence={'Win or Lose': 'win'})
print(q3)

## リスト6.11

In [ ]:
from pyro import sample
from pyro.distributions import Normal

def linear_gaussian():
    n_x = sample("N_x", Normal(9., 3.))
    n_y = sample("N_y", Normal(9., 3.))
    x = 10. + n_x
    y = 2. * x + n_y
    return x, y

## リスト6.12

In [ ]:
from pyro import sample
from pyro.distributions import Gamma

def LiNGAM():
    n_x = sample("N_x", Gamma(9., 1.))
    n_y = sample("N_y", Gamma(9., 1.))
    x = 10. + n_x
    y = 2. * x + n_y
    return x, y

## リスト6.13

In [ ]:
from torch import nn

class EnzymeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.β = nn.Parameter(torch.randn(1, 1))

    def forward(self, x):
        x = torch.mul(x, self.β)
        x = x.log().sigmoid()
        x = torch.mul(x, 100.)
        return x

## リスト6.14

In [ ]:
import pandas as pd
from torch import tensor
import torch

df = pd.read_csv("https://raw.githubusercontent.com/altdeep/causalAI/master/datasets/enzyme-data.csv")
X = torch.tensor(df['x'].values).unsqueeze(1).float()
Y = torch.tensor(df['y'].values).unsqueeze(1).float()

def train(X, Y, model, loss_function, optim, num_epochs):
    loss_history = []
    for epoch in range(num_epochs):
        Y_pred = model(X)
        loss = loss_function(Y_pred, Y)
        loss.backward()
        optim.step()
        optim.zero_grad()
        if epoch % 1000 == 0:
            print(round(loss.data.item(), 6))

torch.manual_seed(1)
enzyme_model = EnzymeModel()
optim = torch.optim.Adam(enzyme_model.parameters(), lr=0.00001)
loss_function = nn.MSELoss()

train(X, Y, enzyme_model, loss_function, optim, num_epochs=60000)

## リスト6.15

In [ ]:
import pyro
from pyro.distributions import Beta, Normal, Uniform
from pyro.infer.mcmc import NUTS, MCMC

def g(u):
  return u / (1 + u)

def model(N):
    β = pyro.sample("β", Beta(0.5, 5.0))
    with pyro.plate("data", N):
        x = pyro.sample("X", Uniform(0.0, 101.0))
        y = pyro.sample("Y", Normal(100.0 * g(β * x), x**.5))
    return x, y

conditioned_model = pyro.condition(
    model,
    data={"X": X.squeeze(1), "Y":  Y.squeeze(1)}
)

N = X.shape[0]
pyro.set_rng_seed(526)
nuts_kernel = NUTS(conditioned_model, adapt_step_size=True)
mcmc = MCMC(nuts_kernel, num_samples=1500, warmup_steps=500)
mcmc.run(N)

## リスト6.16

In [ ]:
from pyro.distributions.transforms import conditional_spline
print(conditional_spline(input_dim=1, context_dim=1))

## リスト6.17

In [ ]:
from pyro.distributions import TransformedDistribution
from pyro.distributions.transforms import AffineTransform


NxDist = Uniform(torch.zeros(1), torch.ones(1))
f_x = AffineTransform(loc=1., scale=100.0)
XDist = TransformedDistribution(NxDist, [f_x])

## リスト6.18

In [ ]:
import pyro
from pyro.distributions import (
    ConditionalTransformedDistribution,
    Normal, Uniform,
    TransformedDistribution
)
from pyro.distributions.transforms import (
    conditional_spline, spline
)
import torch
from torch.distributions.transforms import AffineTransform

pyro.set_rng_seed(348)

NxDist = Uniform(torch.zeros(1), torch.ones(1))
f_x = AffineTransform(loc=1., scale=100.0)
XDist = TransformedDistribution(NxDist, [f_x])

NyDist = Normal(torch.zeros(1), torch.ones(1))
f_y = conditional_spline(input_dim=1, context_dim=1)
YDist = ConditionalTransformedDistribution(NyDist, [f_y])

## リスト6.19

In [ ]:
import matplotlib.pyplot as plt

modules = torch.nn.ModuleList([f_y])
optimizer = torch.optim.Adam(modules.parameters(), lr=3e-3)
losses = []
maxY = max(Y)
Ynorm = Y / maxY
for step in range(800):
    optimizer.zero_grad()
    log_prob_x = XDist.log_prob(X)
    log_prob_y = YDist.condition(X).log_prob(Ynorm)
    loss = -(log_prob_x + log_prob_y).mean()
    loss.backward()
    optimizer.step()
    XDist.clear_cache()
    YDist.clear_cache()
    losses.append(loss.item())

plt.plot(losses[1:])
plt.title("Loss")
plt.xlabel("step")
plt.ylabel("loss")

## リスト6.20

In [ ]:
x_flow = XDist.sample(torch.Size([100,]))
y_flow = YDist.condition(x_flow).sample(torch.Size([100,])) * maxY

plt.title("""
Observed values of enzyme concentration X\n
and protein concentration Y""")
plt.xlabel('X')
plt.ylabel('Y')
plt.xlim(0, 105)
plt.ylim(0, 120)
plt.scatter(
    X.squeeze(1), Y.squeeze(1), color='firebrick',
    label='Actual Data',
    alpha=0.5
)
plt.scatter(
    x_flow.squeeze(1), y_flow.squeeze(),
    label='Generated values from trained model',
    alpha=0.5
)
plt.legend()
plt.show()